In [10]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
    r"G:\My Drive\Spacesmith and Wordsmith's Tower\Spacesmith's HQ"
    r"\Nuclear Energy and Propulsion Engineering\Accenture AI-ML Computational Scientist"
    r"\Credentials\gen-lang-client-0137385761-b0d89e37e8e5.json"
)

In [11]:
from google.cloud import bigquery
from google.oauth2 import service_account

# Google Cloud credentials
credentials = service_account.Credentials.from_service_account_file(

    r"G:\My Drive\Spacesmith and Wordsmith's Tower\Spacesmith's HQ\Nuclear Energy and Propulsion Engineering\Accenture AI-ML Computational Scientist\Credentials\gen-lang-client-0137385761-b0d89e37e8e5.json"

)

In [12]:
from google.cloud import bigquery

# Create a "Client" object
client = bigquery.Client()

# Construct a reference to the "chicago_taxi_trips" dataset
dataset_ref = client.dataset("chicago_taxi_trips", project="bigquery-public-data")

# API request - fetch the dataset
dataset = client.get_dataset(dataset_ref)

In [13]:
# List all tables in the dataset
tables = list(client.list_tables(dataset))

for table in tables:
    print(table.table_id)

taxi_trips


In [14]:
# Construct a reference to the "taxi_trips" table
table_ref = dataset_ref.table("taxi_trips")

# API request - fetch the table
table = client.get_table(table_ref)

# Preview the first five lines of the table
client.list_rows(table, max_results=5).to_dataframe()

,unique_key,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_census_tract,dropoff_census_tract,pickup_community_area,dropoff_community_area,...,extras,trip_total,payment_type,company,pickup_latitude,pickup_longitude,pickup_location,dropoff_latitude,dropoff_longitude,dropoff_location
0,b99416692f5bc01cdb444cbcec7e1105d30d455d,8dd990653cc2793d96b80ae45e928afc9a590b21b491ee...,2019-05-29 18:00:00+00:00,2019-05-29 18:30:00+00:00,1500,1.50,<NA>,<NA>,<NA>,<NA>,...,0.0,12.36,Credit Card,KOAM Taxi Association,NaN,NaN,NaN,NaN,NaN,NaN
1,b99887012c8beccc1dae3f6fcde2fc94ff0ef7b1,d156d4adff01addd62b65753ce67dc8e04bdaf60f7eb14...,2019-05-20 19:15:00+00:00,2019-05-20 19:30:00+00:00,506,1.67,<NA>,<NA>,<NA>,<NA>,...,0.0,10.50,Credit Card,Chicago Carriage Cab Corp,NaN,NaN,NaN,NaN,NaN,NaN
2,b98b6b7afe1f8065d8ba71cd023d9a13f2460f84,beefd3462e3f5a8e854942a2796876f6db73ebbd25b435...,2019-05-08 00:00:00+00:00,2019-05-08 00:00:00+00:00,60,0.00,<NA>,<NA>,<NA>,<NA>,...,0.0,3.25,Cash,Taxi Affiliation Services,NaN,NaN,NaN,NaN,NaN,NaN
3,b9865af3673441b1a79973ad08bd9909bd422af4,ef249ea2c31ccb20ce9eb258d3116254a6f9fe05c7cc84...,2019-05-07 16:30:00+00:00,2019-05-07 17:00:00+00:00,1122,2.86,<NA>,<NA>,<NA>,<NA>,...,1.0,13.25,Cash,Taxi Affiliation Service Yellow,NaN,NaN,NaN,NaN,NaN,NaN
4,e04dbd9a8ba723cc785c4b0398106b72e16d7870,45002471b85d61e62cf602a15145d2bff912e080f50470...,2013-05-16 08:15:00+00:00,2013-05-16 08:15:00+00:00,<NA>,0.00,<NA>,<NA>,<NA>,<NA>,...,0.0,30.06,Credit Card,Chicago Elite Cab Corp.,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
# Print each column name and its data type from the table schema
for field in table.schema:
    print(field.name, "-", field.field_type)

unique_key - STRING
taxi_id - STRING
trip_start_timestamp - TIMESTAMP
trip_end_timestamp - TIMESTAMP
trip_seconds - INTEGER
trip_miles - FLOAT
pickup_census_tract - INTEGER
dropoff_census_tract - INTEGER
pickup_community_area - INTEGER
dropoff_community_area - INTEGER
fare - FLOAT
tips - FLOAT
tolls - FLOAT
extras - FLOAT
trip_total - FLOAT
payment_type - STRING
company - STRING
pickup_latitude - FLOAT
pickup_longitude - FLOAT
pickup_location - STRING
dropoff_latitude - FLOAT
dropoff_longitude - FLOAT
dropoff_location - STRING


In [15]:
# Your code goes here
# EXTRACT(YEAR FROM ...) pulls just the year portion from the timestamp column.
# COUNT(1) counts every row in each group (i.e. each trip).
# GROUP BY year collapses all rows for the same year into one summary row.
# ORDER BY year sorts the results chronologically.
rides_per_year_query = """
        SELECT EXTRACT(YEAR FROM trip_start_timestamp) AS year,
               COUNT(1) AS num_trips
        FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
        GROUP BY year
        ORDER BY year
        """

# Set up the query (cancel the query if it would use too much of 
# your quota)
# QueryJobConfig lets us set a byte limit; if the query would scan more
# than 10 GB (10**10 bytes) BigQuery cancels it before charging us.
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**10)

# client.query() sends the SQL to BigQuery and returns a QueryJob object.
# Passing job_config enforces the byte-billed limit defined above.
rides_per_year_query_job = client.query(rides_per_year_query, job_config=safe_config) # Your code goes here

# API request - run the query, and return a pandas DataFrame
# .to_dataframe() waits for the job to finish, then downloads the results
# as a pandas DataFrame. create_bqstorage_client=False avoids the
# BigQuery Storage API (which requires extra permissions) and uses the
# standard REST API instead.
rides_per_year_result = rides_per_year_query_job.to_dataframe(create_bqstorage_client=False) # Your code goes here

# View results
print(rides_per_year_result)

    year  num_trips
0   2013   27223578
1   2014   37395079
2   2015   32385527
3   2016   31756403
4   2017   24979611
5   2018   20731105
6   2019   17926150
7   2020    3888831
8   2021    3947677
9   2022    6382071
10  2023    6495415


In [17]:
# Your code goes here
# EXTRACT(MONTH FROM ...) pulls the month number (1–12) from the timestamp.
# WHERE EXTRACT(YEAR FROM ...) = 2016 filters rows to only 2016 trips.
# GROUP BY month and ORDER BY month summarise and sort by month number.
rides_per_month_query = """
        SELECT EXTRACT(MONTH FROM trip_start_timestamp) AS month,
               COUNT(1) AS num_trips
        FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
        WHERE EXTRACT(YEAR FROM trip_start_timestamp) = 2016
        GROUP BY month
        ORDER BY month
        """

# Set up the query
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**10)
rides_per_month_query_job = client.query(rides_per_month_query, job_config=safe_config) # Your code goes here

# API request - run the query, and return a pandas DataFrame
rides_per_month_result = rides_per_month_query_job.to_dataframe(create_bqstorage_client=False) # Your code goes here

# View results
print(rides_per_month_result)

    month  num_trips
0       1    2510389
1       2    2568433
2       3    2851106
3       4    2854290
4       5    2859147
5       6    2841872
6       7    2682912
7       8    2629482
8       9    2532650
9      10    2725340
10     11    2387790
11     12    2312992


In [ ]:
# Your code goes here

# --- CTE (Common Table Expression) ---
# WITH ... AS (...) defines a temporary named result set called relevant_rides.
# It runs first and its output is used by the outer SELECT below.
# Selecting only the 3 columns we actually need (hour_of_day, trip_miles,
# trip_seconds) avoids scanning unnecessary data from the wide table.
#
# EXTRACT(HOUR FROM trip_start_timestamp) converts the full timestamp into
# an integer hour (0 = midnight, 23 = 11 PM) so we can group by hour later.
#
# WHERE filters keep only valid, useful rows:
#   - date range > '2016-01-01' AND < '2016-04-01' → Q1 2016 only
#   - trip_seconds > 0  → removes zero-duration trips (invalid data)
#   - trip_miles > 0    → removes zero-distance trips AND prevents
#                         division by zero in the avg_mph calculation
#
# --- Outer SELECT ---
# Reads from the CTE instead of the raw table, so all filters are already applied.
#
# COUNT(1) counts every row in each hour group = total trips that hour.
#
# 3600 * SUM(trip_miles) / SUM(trip_seconds) computes average speed in mph.
#   - SUM(trip_miles) / SUM(trip_seconds) gives miles-per-second for the group.
#   - Multiplying by 3600 (seconds in an hour) converts to miles-per-hour.
#   - Summing miles and seconds separately (rather than averaging per-trip mph)
#     gives a more accurate fleet-wide speed that isn't skewed by short trips.
#
# GROUP BY hour_of_day → one summary row per hour (0–23).
# ORDER BY hour_of_day → results sorted chronologically through the day.

speeds_query = """
        WITH relevant_rides AS (
            SELECT EXTRACT(HOUR FROM trip_start_timestamp) AS hour_of_day,
                   trip_miles,
                   trip_seconds
            FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
            WHERE trip_start_timestamp > '2016-01-01'
              AND trip_start_timestamp < '2016-04-01'
              AND trip_seconds > 0
              AND trip_miles > 0
        )
        SELECT hour_of_day,
               COUNT(1) AS num_trips,
               3600 * SUM(trip_miles) / SUM(trip_seconds) AS avg_mph
        FROM relevant_rides
        GROUP BY hour_of_day
        ORDER BY hour_of_day
        """

# QueryJobConfig caps the bytes BigQuery is allowed to scan.
# If the query would exceed 10 GB (10**10 bytes), it is cancelled automatically
# before any cost is incurred.
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**10)

# client.query() submits the SQL to BigQuery and returns a QueryJob object.
# job_config attaches the byte-billed safety limit to this specific job.
speeds_query_job = client.query(speeds_query, job_config=safe_config) # Your code goes here

# .to_dataframe() blocks until the job completes, then downloads the results
# as a pandas DataFrame.
# create_bqstorage_client=False uses the standard REST API instead of the
# BigQuery Storage API (which requires extra permissions not available here).
speeds_result = speeds_query_job.to_dataframe(create_bqstorage_client=False) # Your code goes here

# View results
print(speeds_result)

    hour_of_day  num_trips    avg_mph
0             0     203092  20.191744
1             1     178046  18.628598
2             2     143447  18.444370
3             3     108899  19.273107
4             4      80067  27.599669
5             5      75786  33.065604
6             6     102254  28.533112
7             7     187585  19.884592
8             8     284223  16.787900
9             9     306854  18.434124
10           10     279762  20.091309
11           11     294006  20.926340
12           12     311522  20.063901
13           13     317225  19.766321
14           14     312629  19.309655
15           15     319953  18.515564
16           16     349455  17.168814
17           17     394324  14.641375
18           18     431991  15.381995
19           19     416743  17.795008
20           20     356279  20.347398
21           21     318363  22.584731
22           22     289886  21.129847
23           23     241690  20.259757


: 